In [8]:
import os
import json
from pathlib import Path
from time import sleep
 
# --- Data ---
import pandas as pd
import numpy as np
 
# --- Image Download ---
import requests
 
# --- CLIP ---
import torch
import clip
from PIL import Image
 
# --- Vector Database ---
import chromadb

In [9]:
CSV_PATH      = "products_combined.csv"     # your product CSV
IMAGE_DIR     = "product_images"       # folder to save images
IMAGE_COL     = "image_url"            # column name in CSV with image URLs
PRODUCT_ID    = "product_id"           # column name for product ID
TITLE_COL     = "title"                # column name for product title
CHROMA_DIR    = "./chromadb_store"     # where ChromaDB saves data
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [10]:
OCCASION_LABELS = [
    "casual everyday wear",
    "party outfit",
    "wedding guest outfit",
    "office wear",
    "date night outfit",
    "beach vacation outfit",
]
 
BOLDNESS_LABELS = [
    "subtle and understated outfit",
    "bold and loud statement outfit",
]
 
COLOR_LABELS = [
    "soft muted pastel colors",
    "bright bold colors",
    "dark moody colors",
    "neutral minimal colors",
]
 
FORMALITY_LABELS = [
    "very formal outfit",
    "semi formal outfit",
    "casual informal outfit",
]

In [11]:
# --- Image Download Function ---
def download_images(df, image_dir=IMAGE_DIR, image_col=IMAGE_COL, product_id_col=PRODUCT_ID):
    """
    Downloads images from URLs and adds local_image_path column to dataframe.
    """
    os.makedirs(image_dir, exist_ok=True)
    local_paths = []
    
    for idx, row in df.iterrows():
        url = row.get(image_col)
        product_id = row.get(product_id_col, idx)
        
        if not url or pd.isna(url):
            print(f"[{idx}] No URL, skipping")
            local_paths.append(None)
            continue
        
        # Images were saved 0-indexed by scrape order (file N.jpg == product_id N+1),
        # so product_id P maps to {P-1}.jpg. See retag_catalog.py.
        ext = url.split('.')[-1].split('?')[0]  # handle URLs with query params
        if ext not in ['jpg', 'jpeg', 'png', 'gif']:
            ext = 'jpg'
        filename = f"{int(product_id) - 1}.{ext}"
        filepath = os.path.join(image_dir, filename)
        
        # Skip if already downloaded
        if os.path.exists(filepath):
            local_paths.append(filepath)
            if idx % 10 == 0:
                print(f"[{idx}] Image exists, skipping download")
            continue
        
        # Download image
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                local_paths.append(filepath)
                if idx % 10 == 0:
                    print(f"[{idx}] Downloaded: {filename}")
            else:
                print(f"[{idx}] Failed to download (status {response.status_code}): {url}")
                local_paths.append(None)
        except Exception as e:
            print(f"[{idx}] Download error: {e}")
            local_paths.append(None)
        
        sleep(0.1)  # Be nice to servers
    
    df["local_image_path"] = local_paths
    return df


In [12]:
def load_clip():
    """Loads CLIP model and preprocessor."""
    print(f"Loading CLIP on {DEVICE}...")
    model, preprocess = clip.load("ViT-B/32", device=DEVICE)
    return model, preprocess
 
 
def score_image_against_labels(image_path, labels, clip_model, preprocess):
    """
    Takes one image and a list of text labels.
    Returns a dict of {label: confidence_score}.
    """
    try:
        image = preprocess(Image.open(image_path).convert("RGB"))
        image = image.unsqueeze(0).to(DEVICE)
        text  = clip.tokenize(labels).to(DEVICE)
 
        with torch.no_grad():
            image_features = clip_model.encode_image(image)
            text_features  = clip_model.encode_text(text)
            similarity     = (image_features @ text_features.T).softmax(dim=-1)
            scores         = similarity[0].cpu().numpy()
 
        return {labels[i]: float(scores[i]) for i in range(len(labels))}
 
    except Exception as e:
        print(f"CLIP error on {image_path}: {e}")
        return None
 
 
def tag_catalog(df, clip_model, preprocess):
    """
    Runs CLIP on every product image.
    Returns dataframe with added tag score columns.
    """
 
    occasion_scores   = []
    boldness_scores   = []
    color_scores      = []
    formality_scores  = []
 
    for idx, row in df.iterrows():
        image_path = row.get("local_image_path")
 
        # Skip if image missing
        if not image_path or not os.path.exists(image_path):
            print(f"[{idx}] No image found, skipping")
            occasion_scores.append(None)
            boldness_scores.append(None)
            color_scores.append(None)
            formality_scores.append(None)
            continue
 
        occasion  = score_image_against_labels(image_path, OCCASION_LABELS,  clip_model, preprocess)
        boldness  = score_image_against_labels(image_path, BOLDNESS_LABELS,  clip_model, preprocess)
        color     = score_image_against_labels(image_path, COLOR_LABELS,     clip_model, preprocess)
        formality = score_image_against_labels(image_path, FORMALITY_LABELS, clip_model, preprocess)
 
        occasion_scores.append(json.dumps(occasion))
        boldness_scores.append(json.dumps(boldness))
        color_scores.append(json.dumps(color))
        formality_scores.append(json.dumps(formality))
 
        if idx % 10 == 0:
            print(f"Tagged {idx}/{len(df)} products")
 
    df["occasion_scores"]  = occasion_scores
    df["boldness_scores"]  = boldness_scores
    df["color_scores"]     = color_scores
    df["formality_scores"] = formality_scores
 
    return df
 

In [ ]:
def store_in_chromadb(df, chroma_dir=CHROMA_DIR):
    """
    Stores all tagged products in ChromaDB for fast retrieval.
    Uses product title as the document text for similarity search.
    """
 
    client     = chromadb.PersistentClient(path=chroma_dir)
 
    # Delete existing collection if re-running
    try:
        client.delete_collection("fashion_products")
    except:
        pass
 
    collection = client.create_collection("fashion_products")
 
    ids       = []
    documents = []
    metadatas = []
 
    for idx, row in df.iterrows():
 
        # Skip rows with no CLIP tags
        if row.get("occasion_scores") is None:
            continue
 
        ids.append(str(row.get(PRODUCT_ID, idx)))
        documents.append(str(row.get(TITLE_COL, "")))
 
        metadatas.append({
            "title":            str(row.get(TITLE_COL, "")),
            "price":            str(row.get("price", "")),
            "category":         str(row.get("category", "")),
            "local_image_path": str(row.get("local_image_path", "")),
            "occasion_scores":  str(row.get("occasion_scores",  "{}")),
            "boldness_scores":  str(row.get("boldness_scores",  "{}")),
            "color_scores":     str(row.get("color_scores",     "{}")),
            "formality_scores": str(row.get("formality_scores", "{}")),
        })
 
    # Store in batches of 100
    batch_size = 100
    for i in range(0, len(ids), batch_size):
        collection.add(
            ids       = ids[i:i+batch_size],
            documents = documents[i:i+batch_size],
            metadatas = metadatas[i:i+batch_size],
        )
 
    print(f"Stored {len(ids)} products in ChromaDB at {chroma_dir}")
    return collection

In [14]:
def query_catalog(user_prompt, top_k=5, chroma_dir=CHROMA_DIR):
    """
    Simple query function to test your stored catalog.
    Pass a natural language prompt, get back matching products.
    """
 
    client     = chromadb.PersistentClient(path=chroma_dir)
    collection = client.get_collection("fashion_products")
 
    results = collection.query(
        query_texts=[user_prompt],
        n_results=top_k,
    )
 
    print(f"\nTop {top_k} results for: '{user_prompt}'\n")
 
    for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
        print(f"{i+1}. {meta['title']}")
        print(f"   Price    : {meta['price']}")
        print(f"   Category : {meta['category']}")
        print(f"   Image    : {meta['local_image_path']}")
        print()
 
    return results

In [15]:
if __name__ == "__main__":
 
    # --- Load CSV ---
    print("Loading catalog CSV...")
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df)} products")
 
    # --- Step 1: Download Images ---
    print("\n[Step 1] Downloading images...")
    df = download_images(df)
    print(f"Downloaded {df['local_image_path'].notna().sum()} images")
 
    # --- Step 2: CLIP Tagging ---
    print("\n[Step 2] Loading CLIP and tagging products...")
    clip_model, preprocess = load_clip()
    df = tag_catalog(df, clip_model, preprocess)
    df.to_csv("tagged_catalog.csv", index=False)   # save tagged version
    print("Tagging complete. Saved to tagged_catalog.csv")
 
    # --- Step 3: Store in ChromaDB ---
    print("\n[Step 3] Storing in ChromaDB...")
    collection = store_in_chromadb(df)
 
    # --- Step 4: Test Query ---
    print("\n[Step 4] Testing a query...")
    query_catalog("friend's birthday party, want to look nice but not steal attention")

Loading catalog CSV...
Loaded 296 products

[Step 1] Downloading images...
[0] Image exists, skipping download
[10] Image exists, skipping download
[20] Image exists, skipping download
[30] Image exists, skipping download
[40] Image exists, skipping download
[50] Image exists, skipping download
[60] Image exists, skipping download
[70] Image exists, skipping download
[80] Image exists, skipping download
[90] Image exists, skipping download
[100] Image exists, skipping download
[110] Image exists, skipping download
[120] Image exists, skipping download
[130] Image exists, skipping download
[140] Image exists, skipping download
[150] Image exists, skipping download
[160] Image exists, skipping download
[170] Image exists, skipping download
[180] Image exists, skipping download
[190] Image exists, skipping download
[200] Image exists, skipping download
[210] Image exists, skipping download
[220] Image exists, skipping download
[230] Image exists, skipping download
[240] Image exists, skip

In [16]:
# This is OpenAI's CLIP — the one we need
# subprocess.run(["pip", "install", "git+https://github.com/openai/CLIP.git"])

In [19]:
df["title"].value_counts()

title
Straight Fit Cargo Jeans                     3
Red Draped Detail Mini Dress                 2
White Typography Crew Neck T-Shirt           2
Black Typography Crew Neck T-Shirt           2
Black Floral Buttoned Shirt                  2
                                            ..
Gold Imitation Beaded Layered Necklace       1
Gold Imitation Moon Star Pendant Necklace    1
Imitation Gold Chain Link Bracelet Set       1
Golden Chic Hoop Earrings                    1
Golden Set Of 6 Hoop Earrings                1
Name: count, Length: 283, dtype: int64

In [21]:
import pandas as pd
import json

df = pd.read_csv("tagged_catalog.csv")
print(f"Total products: {len(df)}")
print("\nProduct titles sample:")
for title in df["title"].tolist():
    print(title)

Total products: 296

Product titles sample:
Blue Mid Rise Bootcut Pants
Off White Striped Maxi Dress
White Polka Dot One Shoulder Top
Red Draped Detail Mini Dress
Black Floral V-Neck Mini Dress
Maroon One Shoulder Ruched Maxi Dress
Maroon Lace Bodycon Maxi Dress
Black Pu Slit Bodycon Skirt
Yellow Floral A-Line Maxi Dress
Brown Solid V-Neck Top
Maroon Embellished Off Shoulder Dress
Black Solid One Shoulder Top
Maroon Solid Slit Bodycon Midi Dress
Red Solid Fitted Maxi Dress
Yellow Floral A-Line Mini Dress
Red One Shoulder Ruched Dress
Dark Grey Mid Rise Wide Led Jeans
Beige Sweetheart Mermaid Maxi Dress
Brown High-Rise Bootcut Trousers
White Solid Halter Neck Top
Maroon Ribbed Mock Neck Mini Dress
Off White Solid Halter Neck Mini Dress
Zipper Up Denim Skirt With Flap Pockets
Beige Ruched One-Shoulder Mini Dress
Brown Abstract Cowl Neck Midi Dress
Black High-Rise Striped Bootcut Pants
Rust Sweetheart Ruffle Mini Dress
Spiderman Inspired Oversized Zipper Hoodie
Wine Sequin Mini Corset Dre

In [22]:
import pandas as pd
import json

df = pd.read_csv("tagged_catalog.csv")

# Remove accessories
accessory_keywords = ["necklace", "earring", "bracelet", "ring", "chain", 
                      "pendant", "choker", "waist chain", "hair pin", "jewellery"]

mask = df["title"].str.lower().apply(
    lambda t: not any(kw in t for kw in accessory_keywords)
)

df_clean = df[mask]
print(f"Before: {len(df)} | After: {len(df_clean)} | Removed: {len(df) - len(df_clean)}")
df_clean.to_csv("tagged_catalog.csv", index=False)

Before: 296 | After: 263 | Removed: 33


In [23]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_DIR)

# Delete old collection
try:
    client.delete_collection("fashion_products")
    print("Old collection deleted.")
except:
    pass

# Re-store clean catalog
collection = client.create_collection("fashion_products")

for idx, row in df_clean.iterrows():
    if row.get("occasion_scores") is None:
        continue
    collection.add(
        ids=[str(idx)],
        documents=[str(row.get("title", ""))],
        metadatas=[{
            "title":            str(row.get("title", "")),
            "price":            str(row.get("price", "")),
            "category":         str(row.get("category", "")),
            "local_image_path": str(row.get("local_image_path", "")),
            "occasion_scores":  str(row.get("occasion_scores", "{}")),
            "boldness_scores":  str(row.get("boldness_scores", "{}")),
            "color_scores":     str(row.get("color_scores", "{}")),
            "formality_scores": str(row.get("formality_scores", "{}")),
        }]
    )

print(f"Stored {collection.count()} products.")

Old collection deleted.
Stored 263 products.


In [24]:
df = pd.read_csv("tagged_catalog.csv")

df["office_score"] = df["occasion_scores"].apply(
    lambda x: json.loads(x).get("office wear", 0) if isinstance(x, str) else 0
)

print(df[["title", "office_score"]]
    .sort_values("office_score", ascending=False)
    .head(15)
    .to_string())

                                             title  office_score
260               Black Solid High Rise Wrap Pants      0.990383
258                 White High Rise Wide Leg Pants      0.987710
173              Light Yellow Solid Buttoned Shirt      0.959316
161  Light Brown Plaid Oversized Button Down Shirt      0.948531
149                    Black Floral Buttoned Shirt      0.936337
56                     Black Floral Buttoned Shirt      0.936337
162                Beige Cinched Waist Plaid Shirt      0.665042
165                   White Lace Band Collar Shirt      0.660659
171      Beige Oversized Striped Button Down Shirt      0.656374
1                     Off White Striped Maxi Dress      0.651473
156                     White Solid Buttoned Shirt      0.643757
158                   Yellow Striped Regular Shirt      0.643192
99               Off White Solid Buttoned Jumpsuit      0.578183
168          Multi Color Striped Button Down Shirt      0.558394
237                Grey M

In [27]:
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection("fashion_products")

# Query directly for office wear
results = collection.query(
    query_texts=["office wear formal work outfit"],
    n_results=10
)

for meta in results["metadatas"][0]:
    print(meta.get("title"))

Grey Mid Rise Wrap Formal Pants
Off White Striped Maxi Dress
Light Pink Lace Sheer Cutout Neck Top
Off White Solid Buttoned Jumpsuit
White Solid Fitted Buttoned Shirt
White Solid Buttoned Shirt
Off White Solid Halter Neck Mini Dress
Beige Striped Oversized Shirt
Yellow Striped Regular Shirt
White Fitted Sports Jumpsuit


In [26]:
# Rebuild ChromaDB with richer document text
client = chromadb.PersistentClient(path=CHROMA_DIR)
try:
    client.delete_collection("fashion_products")
except:
    pass

collection = client.create_collection("fashion_products")

df = pd.read_csv("tagged_catalog.csv")

for idx, row in df_clean.iterrows():
    if not isinstance(row.get("occasion_scores"), str):
        continue

    occ_scores  = json.loads(row["occasion_scores"])
    bold_scores = json.loads(row["boldness_scores"])

    # Find top occasion label
    top_occasion = max(occ_scores, key=occ_scores.get)
    top_boldness = max(bold_scores, key=bold_scores.get)

    # Richer document = better retrieval
    document = f"{row['title']} {top_occasion} {top_boldness}"

    collection.add(
        ids=[str(idx)],
        documents=[document],      # ← now includes occasion context
        metadatas=[{
            "title":            str(row.get("title", "")),
            "price":            str(row.get("price", "")),
            "category":         str(row.get("category", "")),
            "local_image_path": str(row.get("local_image_path", "")),
            "occasion_scores":  str(row.get("occasion_scores", "{}")),
            "boldness_scores":  str(row.get("boldness_scores", "{}")),
            "color_scores":     str(row.get("color_scores",    "{}")),
            "formality_scores": str(row.get("formality_scores","{}")),
        }]
    )

print(f"Rebuilt ChromaDB with {collection.count()} products.")

Rebuilt ChromaDB with 263 products.
